# Bronze Layer — Standalone Ingestion

Run each pipeline cell individually to sink data into `health_catalog.bronze.*`.
Use this to test and debug each source independently.

In [ ]:
import os
import pandas as pd
from pyspark.sql import SparkSession
from databricks.sdk import WorkspaceClient

spark = SparkSession.builder.getOrCreate()

DATABRICKS_HOST = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN = os.getenv('DATABRICKS_TOKEN')

client = WorkspaceClient(host=DATABRICKS_HOST, token=DATABRICKS_TOKEN)

catalog = 'health_catalog'
schema = 'bronze'

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

def write_table(table_name: str, df: pd.DataFrame):
    if df.empty:
        print(f"  Skipping {catalog}.{schema}.{table_name}: empty")
        return
    spark_df = spark.createDataFrame(df)
    full_name = f"{catalog}.{schema}.{table_name}"
    spark_df.write.mode("overwrite").saveAsTable(full_name)
    print(f"  Wrote {len(df)} rows to {full_name}")

### 1. Databricks REST API

In [ ]:
from databricks_health.ingestors import DatabricksApiIngestor

api = DatabricksApiIngestor(client, lookback_days=30)
api_data = api.ingest()
for name, df in api_data.items():
    print(f"{name}: {len(df)} rows")
    write_table(name, df)

### 2. System Tables

In [ ]:
from databricks_health.ingestors import SystemTablesIngestor

system = SystemTablesIngestor(spark, lookback_days=30)
system_data = system.ingest()
for name, df in system_data.items():
    print(f"{name}: {len(df)} rows")
    write_table(name, df)

### 3. Azure Monitor (optional)

In [ ]:
AZURE_LOG_ANALYTICS_WORKSPACE_ID = os.getenv('AZURE_LOG_ANALYTICS_WORKSPACE_ID')

if AZURE_LOG_ANALYTICS_WORKSPACE_ID:
    from azure.identity import DefaultAzureCredential
    from azure.monitor.query import LogsQueryClient
    from databricks_health.ingestors import AzureMonitorIngestor

    credential = DefaultAzureCredential()
    logs_client = LogsQueryClient(credential)
    azure = AzureMonitorIngestor(logs_client, AZURE_LOG_ANALYTICS_WORKSPACE_ID, lookback_days=30)
    azure_data = azure.ingest()
    for name, df in azure_data.items():
        print(f"{name}: {len(df)} rows")
        write_table(name, df)
else:
    print("AZURE_LOG_ANALYTICS_WORKSPACE_ID not set — skipping Azure Monitor")